# 14.1 怎麼表示相對位置？


「小貓追小狗」與「小狗追小貓」包含同樣的詞，意思卻不同。注意力需要知道詞的先後，也常需要知道兩個詞相隔幾格。如果在句首加上「今天」，貓與狗的位置編號都增加一，但彼此距離並沒有變；我們希望用來比較兩者的某部分規則仍然相同。先回顧[注意力中的比對](https://birdhackor.github.io/tiny-perceptron-vlm/3.2.html)與[位置資訊](https://birdhackor.github.io/tiny-perceptron-vlm/4.1.html)：Query（Q）是當前位置提出的查詢向量，Key（K）是另一位置供它比對的向量，兩者點積提供注意力分數。

一個做法是把向量中的數字兩兩分組，當成平面上的箭頭。例如 `[1,0]` 是朝右、長度為一的箭頭。位置每往後一格，就把箭頭旋轉固定角度。假設每格轉45度，位置2的Q與位置5的K相差三格，旋轉角度差就是135度。若兩者都往後移十格，共同多轉450度，角度差仍是135度。箭頭長度也沒有變，因此兩者點積保持相同。這種把位置寫進旋轉的方式叫 Rotary Position Embedding，常縮寫為 RoPE。

下圖用一對數字形成的Q、K箭頭示意。圖中的45度是兩箭頭當下的夾角；一起轉動時夾角不變。實際模型有許多對數字，不同對使用不同轉速，並非所有位置都只用一個45度鐘面。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/rope.svg")))

下面只用兩個數字，讓我們親手核對旋轉性質。`rotate` 接受一支箭頭及位置編號，`math.cos`、`math.sin` 依旋轉公式計算新座標；角度單位是弧度，`pi / 4` 等於45度。`score` 把兩支旋轉後的箭頭逐項相乘再相加，也就是點積。


In [ ]:
import math
import torch


def rotate(x, position):
    angle = position * math.pi / 4
    c, s = math.cos(angle), math.sin(angle)
    return torch.tensor([c * x[0] - s * x[1], s * x[0] + c * x[1]])


q = torch.tensor([1.0, 0.0])
k = torch.tensor([1.0, 0.0])


def score(a, b):
    return torch.dot(rotate(q, a), rotate(k, b)).item()


print(round(score(2, 5), 4))
print(round(score(12, 15), 4))
print(round(score(3, 5), 4))

前兩行都是 `-0.7071`，正是135度的餘弦；第三行是 `0.0`，因為只移Q，距離由三格變成兩格，角度差變成90度。負點積是比對分數，不是負機率，後續仍要透過softmax轉成注意力權重。這個實驗證明共同平移保留旋轉點積，不能證明模型在訓練未見過的超長文章上仍答得好：多種轉速與內容一起作用，還需要長度測試。

練習把第三行改成 `score(22, 25)`，先預測是否等於前兩行，再執行核對；接著改成 `score(22, 26)`。你應能用「相隔三格或四格」解釋結果，而不用重新記一個位置表。
